In [1]:
from pathlib import Path

Path("mcp_servers").mkdir(exist_ok=True)  # all the server code of this chapter goes into this folder

In [2]:
%%writefile mcp_servers/tools.py
import inspect
from typing import Annotated, Callable, List, get_args, get_origin, get_type_hints


class ToolCall:

    def __init__(self, id: str, name: str, arguments: dict):
        self.id = id
        self.name = name
        self.arguments = arguments

    def to_dict(self) -> dict:
        return {"id": self.id, "name": self.name, "arguments": self.arguments}

    def __repr__(self) -> str:
        return f"{self.name}({json.dumps(self.arguments, ensure_ascii=False)})"


class Tool:

    def __init__(self, name: str, description: str, parameters: dict, function: Callable):
        self.name = name
        self.description = description
        self.parameters = parameters  # JSON Schema of the arguments
        self.function = function

    def run(self, arguments: dict) -> str:
        return str(self.function(**arguments))

    def __call__(self, *args, **kwargs):
        return self.function(*args, **kwargs)


JSON_TYPES = {str: "string", int: "integer", float: "number", bool: "boolean", list: "array", dict: "object"}


def tool(function: Callable) -> Tool:
    """Decorator: build a Tool from a function's name, docstring and type hints."""
    hints = get_type_hints(function, include_extras=True)
    properties, required = {}, []

    for name, param in inspect.signature(function).parameters.items():
        hint, description = hints.get(name, str), None
        if get_origin(hint) is Annotated:  # Annotated[int, "description"]
            hint, description = get_args(hint)[:2]
        properties[name] = {"type": JSON_TYPES.get(hint, "string")}
        if description:
            properties[name]["description"] = description
        if param.default is inspect.Parameter.empty:
            required.append(name)

    return Tool(
        name=function.__name__,
        description=inspect.getdoc(function) or "",
        parameters={"type": "object", "properties": properties, "required": required},
        function=function
    )


class ToolRegistry:

    def __init__(self, tools: List[Tool] = None):
        self.tools = {}
        for t in tools or []:
            self.register(t)

    def register(self, t: Tool) -> Tool:
        if t.name in self.tools:
            raise ValueError(f"duplicate tool name: '{t.name}'")
        self.tools[t.name] = t
        return t

    def list(self) -> List[Tool]:
        return list(self.tools.values())

    def run(self, call: ToolCall) -> str:
        t = self.tools.get(call.name)
        if t is None:
            return f"Error: unknown tool '{call.name}'"
        try:
            return t.run(call.arguments)
        except Exception as e:
            return f"Error: {e}"  # let the model see the error and try to fix it


@tool
def get_current_time(timezone: Annotated[str, "IANA timezone name, e.g. 'America/Toronto', 'Asia/Shanghai'"]) -> str:
    """Get the current local date and time in a given IANA timezone."""
    return datetime.now(ZoneInfo(timezone)).strftime("%Y-%m-%d %H:%M:%S %Z")


@tool
def add_days(date: Annotated[str, "Start date in YYYY-MM-DD format"],
             days: Annotated[int, "Number of days to add, negative to subtract"] = 1) -> str:
    """Add (or subtract, if negative) a number of days to a date. Returns the new date and its weekday."""
    result = datetime.strptime(date, "%Y-%m-%d") + timedelta(days=days)
    return result.strftime("%Y-%m-%d (%A)")


registry = ToolRegistry([get_current_time, add_days])


Writing mcp_servers/tools.py


In [3]:
%%writefile mcp_servers/mcp_server_raw.py
import json
import sys

from tools import ToolCall, registry

PROTOCOL_VERSION = "2025-06-18"


def log(text: str):
    print(f"[server] {text}", file=sys.stderr, flush=True)  # stdout is reserved for the protocol!


def handle(method: str, params: dict) -> dict:
    if method == "initialize":
        return {
            "protocolVersion": PROTOCOL_VERSION,
            "capabilities": {"tools": {}},
            "serverInfo": {"name": "raw-time-server", "version": "0.1"}
        }
    if method == "tools/list":
        return {"tools": [
            {"name": t.name, "description": t.description, "inputSchema": t.parameters}
            for t in registry.list()
        ]}
    if method == "tools/call":
        result = registry.run(ToolCall("", params["name"], params.get("arguments", {})))
        return {"content": [{"type": "text", "text": result}], "isError": result.startswith("Error")}
    if method == "ping":
        return {}
    raise LookupError(f"Method not found: {method}")


for line in sys.stdin:  # one JSON-RPC message per line
    request = json.loads(line)
    log(f"<- {request['method']}")
    if "id" not in request:  # a notification (e.g. notifications/initialized) gets no response
        continue
    try:
        response = {"jsonrpc": "2.0", "id": request["id"], "result": handle(request["method"], request.get("params", {}))}
    except LookupError as e:
        response = {"jsonrpc": "2.0", "id": request["id"], "error": {"code": -32601, "message": str(e)}}
    print(json.dumps(response, ensure_ascii=False), flush=True)


Writing mcp_servers/mcp_server_raw.py


In [4]:
import json
import subprocess
import sys


class RawMCPClient:

    def __init__(self, command: list):
        self.process = subprocess.Popen(command, stdin=subprocess.PIPE, stdout=subprocess.PIPE,
                                        stderr=subprocess.PIPE, text=True)
        self.next_id = 1

    def send(self, method: str, params: dict = None, notification: bool = False):
        message = {"jsonrpc": "2.0", "method": method}
        if params is not None:
            message["params"] = params
        if not notification:
            message["id"] = self.next_id
            self.next_id += 1

        print("->", json.dumps(message, ensure_ascii=False))
        self.process.stdin.write(json.dumps(message) + "\n")
        self.process.stdin.flush()
        if notification:
            return None

        response = json.loads(self.process.stdout.readline())
        print("<-", json.dumps(response, ensure_ascii=False))
        return response

    def close(self):
        _, stderr = self.process.communicate(timeout=5)  # closes stdin, the server loop ends
        print("server stderr:\n" + stderr)

In [5]:
client = RawMCPClient([sys.executable, "mcp_servers/mcp_server_raw.py"])
client.send("initialize", {
    "protocolVersion": "2025-06-18",
    "capabilities": {},
    "clientInfo": {"name": "raw-client", "version": "0.1"}
})
client.send("notifications/initialized", notification=True)

-> {"jsonrpc": "2.0", "method": "initialize", "params": {"protocolVersion": "2025-06-18", "capabilities": {}, "clientInfo": {"name": "raw-client", "version": "0.1"}}, "id": 1}
<- {"jsonrpc": "2.0", "id": 1, "result": {"protocolVersion": "2025-06-18", "capabilities": {"tools": {}}, "serverInfo": {"name": "raw-time-server", "version": "0.1"}}}
-> {"jsonrpc": "2.0", "method": "notifications/initialized"}


In [6]:
response = client.send("tools/list")

-> {"jsonrpc": "2.0", "method": "tools/list", "id": 2}
<- {"jsonrpc": "2.0", "id": 2, "result": {"tools": [{"name": "get_current_time", "description": "Get the current local date and time in a given IANA timezone.", "inputSchema": {"type": "object", "properties": {"timezone": {"type": "string", "description": "IANA timezone name, e.g. 'America/Toronto', 'Asia/Shanghai'"}}, "required": ["timezone"]}}, {"name": "add_days", "description": "Add (or subtract, if negative) a number of days to a date. Returns the new date and its weekday.", "inputSchema": {"type": "object", "properties": {"date": {"type": "string", "description": "Start date in YYYY-MM-DD format"}, "days": {"type": "integer", "description": "Number of days to add, negative to subtract"}}, "required": ["date"]}}]}}


In [7]:
client.send("tools/call", {"name": "add_days", "arguments": {"date": "2026-11-11", "days": 100}})
client.send("tools/call", {"name": "get_current_time", "arguments": {"timezone": "Mars/Base"}})

-> {"jsonrpc": "2.0", "method": "tools/call", "params": {"name": "add_days", "arguments": {"date": "2026-11-11", "days": 100}}, "id": 3}
<- {"jsonrpc": "2.0", "id": 3, "result": {"content": [{"type": "text", "text": "2027-02-19 (Friday)"}], "isError": false}}
-> {"jsonrpc": "2.0", "method": "tools/call", "params": {"name": "get_current_time", "arguments": {"timezone": "Mars/Base"}}, "id": 4}
<- {"jsonrpc": "2.0", "id": 4, "result": {"content": [{"type": "text", "text": "Error: 'No time zone found with key Mars/Base'"}], "isError": true}}


{'jsonrpc': '2.0',
 'id': 4,
 'result': {'content': [{'type': 'text',
    'text': "Error: 'No time zone found with key Mars/Base'"}],
  'isError': True}}

In [8]:
client.send("resources/list")
client.close()

-> {"jsonrpc": "2.0", "method": "resources/list", "id": 5}
<- {"jsonrpc": "2.0", "id": 5, "error": {"code": -32601, "message": "Method not found: resources/list"}}
server stderr:
[server] <- initialize
[server] <- notifications/initialized
[server] <- tools/list
[server] <- tools/call
[server] <- tools/call
[server] <- resources/list



In [9]:
%%writefile mcp_servers/mcp_server_sdk.py
from datetime import datetime, timedelta
from typing import Annotated
from zoneinfo import ZoneInfo

from mcp.server.mcpserver import MCPServer
from mcp.server.mcpserver.exceptions import ToolError
from pydantic import Field

mcp = MCPServer("sdk-time-server")


@mcp.tool()
def get_current_time(timezone: Annotated[str, Field(description="IANA timezone name, e.g. 'America/Toronto', 'Asia/Shanghai'")]) -> str:
    """Get the current local date and time in a given IANA timezone."""
    try:
        tz = ZoneInfo(timezone)
    except Exception:
        raise ToolError(f"unknown timezone '{timezone}', use an IANA name such as 'Asia/Tokyo'")
    return datetime.now(tz).strftime("%Y-%m-%d %H:%M:%S %Z")


@mcp.tool()
def add_days(date: Annotated[str, Field(description="Start date in YYYY-MM-DD format")],
             days: Annotated[int, Field(description="Number of days to add, negative to subtract")] = 1) -> str:
    """Add (or subtract, if negative) a number of days to a date. Returns the new date and its weekday."""
    result = datetime.strptime(date, "%Y-%m-%d") + timedelta(days=days)
    return result.strftime("%Y-%m-%d (%A)")


if __name__ == "__main__":
    mcp.run()  # transport="stdio" by default


Writing mcp_servers/mcp_server_sdk.py


In [10]:
client = RawMCPClient([sys.executable, "mcp_servers/mcp_server_sdk.py"])
client.send("initialize", {
    "protocolVersion": "2025-06-18",
    "capabilities": {},
    "clientInfo": {"name": "raw-client", "version": "0.1"}
})
client.send("notifications/initialized", notification=True)

-> {"jsonrpc": "2.0", "method": "initialize", "params": {"protocolVersion": "2025-06-18", "capabilities": {}, "clientInfo": {"name": "raw-client", "version": "0.1"}}, "id": 1}
<- {"jsonrpc": "2.0", "id": 1, "result": {"capabilities": {"prompts": {"listChanged": false}, "resources": {"listChanged": false, "subscribe": false}, "tools": {"listChanged": false}}, "protocolVersion": "2025-06-18", "serverInfo": {"name": "sdk-time-server", "version": ""}}}
-> {"jsonrpc": "2.0", "method": "notifications/initialized"}


In [11]:
response = client.send("tools/list")

-> {"jsonrpc": "2.0", "method": "tools/list", "id": 2}
<- {"jsonrpc": "2.0", "id": 2, "result": {"tools": [{"description": "Get the current local date and time in a given IANA timezone.", "inputSchema": {"properties": {"timezone": {"description": "IANA timezone name, e.g. 'America/Toronto', 'Asia/Shanghai'", "title": "Timezone", "type": "string"}}, "required": ["timezone"], "type": "object", "title": "get_current_timeArguments"}, "name": "get_current_time", "outputSchema": {"properties": {"result": {"title": "Result", "type": "string"}}, "required": ["result"], "type": "object", "title": "get_current_timeOutput"}}, {"description": "Add (or subtract, if negative) a number of days to a date. Returns the new date and its weekday.", "inputSchema": {"properties": {"date": {"description": "Start date in YYYY-MM-DD format", "title": "Date", "type": "string"}, "days": {"default": 1, "description": "Number of days to add, negative to subtract", "title": "Days", "type": "integer"}}, "required": 

In [12]:
client.send("tools/call", {"name": "add_days", "arguments": {"date": "2026-11-11", "days": 100}})
client.send("tools/call", {"name": "get_current_time", "arguments": {"timezone": "Mars/Base"}})

-> {"jsonrpc": "2.0", "method": "tools/call", "params": {"name": "add_days", "arguments": {"date": "2026-11-11", "days": 100}}, "id": 3}
<- {"jsonrpc": "2.0", "id": 3, "result": {"content": [{"text": "2027-02-19 (Friday)", "type": "text"}], "isError": false, "structuredContent": {"result": "2027-02-19 (Friday)"}}}
-> {"jsonrpc": "2.0", "method": "tools/call", "params": {"name": "get_current_time", "arguments": {"timezone": "Mars/Base"}}, "id": 4}
<- {"jsonrpc": "2.0", "id": 4, "result": {"content": [{"text": "Error executing tool get_current_time: unknown timezone 'Mars/Base', use an IANA name such as 'Asia/Tokyo'", "type": "text"}], "isError": true}}


{'jsonrpc': '2.0',
 'id': 4,
 'result': {'content': [{'text': "Error executing tool get_current_time: unknown timezone 'Mars/Base', use an IANA name such as 'Asia/Tokyo'",
    'type': 'text'}],
  'isError': True}}

In [13]:
client.send("resources/list")
client.close()

-> {"jsonrpc": "2.0", "method": "resources/list", "id": 5}
<- {"jsonrpc": "2.0", "id": 5, "result": {"resources": []}}
server stderr:
Tool 'get_current_time' failed: "Error executing tool get_current_time: unknown timezone 'Mars/Base', use an IANA name such as 'Asia/Tokyo'"

